# 04 — Feature Engineering

Este notebook implementa y compara nuevas variables sustentadas en las hipótesis de Feature Analysis. La primera iteración utiliza Logistic Regression como instrumento de comparación controlada: se mantienen fijos el modelo, sus hiperparámetros, los folds y los grupos del baseline.

## Protocolo experimental

- Desarrollo: reservas de 2015–2016.
- Holdout temporal: enero–agosto de 2017. Se separa, pero no se consulta durante la selección de features.
- Validación interna: cinco folds de `StratifiedGroupKFold`.
- Grupos: combinaciones exactas del conjunto base, idénticas para todos los experimentos.
- Modelo fijo: Logistic Regression con `C=10` y `class_weight='balanced'`.
- Métrica principal: ROC AUC medio y desviación entre folds.

Las transformaciones de esta etapa son determinísticas y se calculan por fila: no aprenden estadísticas del target ni del dataset completo. La imputación, codificación y escala permanecen dentro del pipeline.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

pd.set_option("display.max_columns", None)

In [2]:
def find_project_root(start_path=None):
    """Busca la raíz del repositorio a partir del directorio actual."""
    start = Path(start_path or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir() and (candidate / "src").is_dir():
            return candidate
    return None

PROJECT_ROOT = find_project_root()
if PROJECT_ROOT is None:
    raise FileNotFoundError("No se encontró la raíz local del proyecto.")

project_root_str = str(PROJECT_ROOT)
if project_root_str not in sys.path:
    sys.path.insert(0, project_root_str)

print(f"Raíz del proyecto: {PROJECT_ROOT}")

Raíz del proyecto: C:\Code\Vialesoft\Vialesoft_Devlab\Mini_Proyectos\Machine_Learning\Obligatorio2026


In [3]:
from src.config import RANDOM_STATE, RAW_DATA_DIR
from src.data import (
    combine_hotel_datasets,
    load_csv_with_fallback,
    make_temporal_holdout_split,
    normalize_text_values,
)
from src.evaluation import (
    cross_validate_model,
    get_stratified_group_cross_validation,
)
from src.features import (
    add_profile_indicator_features,
    add_stay_composition_features,
    add_waiting_list_interaction_features,
    build_base_feature_set,
    build_first_engineered_feature_set,
    get_feature_types,
    make_exact_feature_groups,
)
from src.preprocessing import build_tabular_preprocessor

## Carga y conjunto base

In [4]:
H1_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H1.csv"
H2_URL = "https://raw.githubusercontent.com/Lithium582/Obligatorio2026/refs/heads/main/data/raw/H2.csv"

h1_df = load_csv_with_fallback(RAW_DATA_DIR / "H1.csv", H1_URL)
h2_df = load_csv_with_fallback(RAW_DATA_DIR / "H2.csv", H2_URL)
bookings_df = normalize_text_values(combine_hotel_datasets(h1_df, h2_df))
X_base, y = build_base_feature_set(bookings_df)

print(f"Dataset: {len(X_base):,} filas")
print(f"Conjunto base: {X_base.shape[1]} features")

Dataset: 119,390 filas
Conjunto base: 25 features


In [5]:
X_train_base, X_holdout_base, y_train, y_holdout = make_temporal_holdout_split(
    X_base, y, X_base["ArrivalDateYear"], validation_period=2017
)
base_groups = make_exact_feature_groups(X_train_base)

split_summary = pd.DataFrame({
    "subset": ["Development 2015-2016", "Reserved holdout 2017 Jan-Aug"],
    "rows": [len(X_train_base), len(X_holdout_base)],
    "cancellation_rate": [y_train.mean(), y_holdout.mean()],
})
display(split_summary.round(4))
print(f"Grupos exactos en desarrollo: {base_groups.nunique():,}")

,subset,rows,cancellation_rate
0,Development 2015-2016,78703,0.3619
1,Reserved holdout 2017 Jan-Aug,40687,0.3870


Grupos exactos en desarrollo: 52,930


# Primer lote de features

## Composición de la reserva

- `TotalGuests = Adults + Children + Babies`.
- `TotalNights = StaysInWeekendNights + StaysInWeekNights`.

Estas sumas aportan una representación directa del tamaño de la reserva y de la duración total, sin reemplazar inicialmente sus componentes.

## Indicadores de perfil e historial

- `IsDomestic`: identifica reservas cuyo país es Portugal (`PRT`).
- `WasOnWaitingList`: separa el cero estructural de los tiempos positivos.
- `HasPreviousCancellation`: distingue clientes con cancelaciones anteriores.
- `PreviousCancellationRate`: cancelaciones anteriores sobre el total de reservas históricas conocidas; vale cero cuando no existe historial.

`IsDomestic` no utiliza el target ni agrupaciones aprendidas. Cuando `Country` es nulo, el indicador también permanece nulo y se imputa dentro del pipeline. La categoría `Country` original se conserva para comprobar si el indicador agrega una representación útil o resulta redundante.

## Interacción de lista de espera y hotel

Feature Analysis mostró que la relación de `DaysInWaitingList` cambia de dirección entre City Hotel y Resort Hotel. `CityHotelWasOnWaitingList` permite que Logistic Regression represente efectos diferentes según el hotel, en lugar de imponer un único coeficiente común.

In [6]:
feature_sets = {
    "Base": X_train_base.copy(),
    "Stay composition": add_stay_composition_features(X_train_base),
    "Profile and history indicators": add_profile_indicator_features(X_train_base),
    "Waiting-list interaction": add_waiting_list_interaction_features(X_train_base),
    "Complete first batch": build_first_engineered_feature_set(X_train_base),
}

base_columns = set(X_train_base.columns)
feature_set_summary = pd.DataFrame([
    {
        "feature_set": name,
        "total_features": frame.shape[1],
        "added_features": sorted(set(frame.columns).difference(base_columns)),
        "missing_values": int(frame.isna().sum().sum()),
    }
    for name, frame in feature_sets.items()
])
display(feature_set_summary)

,feature_set,total_features,added_features,missing_values
0,Base,25,[],409
1,Stay composition,27,"[TotalGuests, TotalNights]",409
2,Profile and history indicators,29,"[HasPreviousCancellation, IsDomestic, Previous...",814
3,Waiting-list interaction,27,"[CityHotelWasOnWaitingList, WasOnWaitingList]",409
4,Complete first batch,32,"[CityHotelWasOnWaitingList, HasPreviousCancell...",814


# Comparación mediante validación agrupada

Cada conjunto recibe un preprocesador construido a partir de sus propias columnas. Las categorías originales permanecen constantes y todas las features nuevas se tratan como numéricas. El holdout temporal no interviene en este bloque.

In [7]:
def build_fixed_logistic_pipeline(X):
    numeric_features, categorical_features = get_feature_types(X)
    preprocessor = build_tabular_preprocessor(
        numeric_features=numeric_features,
        categorical_features=categorical_features,
        scale_numeric=True,
    )
    return Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(
            C=10.0,
            class_weight="balanced",
            solver="liblinear",
            max_iter=2000,
            random_state=RANDOM_STATE,
        )),
    ])

In [8]:
experiment_records = []
fold_records = []

for feature_set_name, X_experiment in feature_sets.items():
    pipeline = build_fixed_logistic_pipeline(X_experiment)
    group_cv = get_stratified_group_cross_validation(n_splits=5)
    scores = cross_validate_model(
        estimator=pipeline,
        X=X_experiment,
        y=y_train,
        scoring="roc_auc",
        cv=group_cv,
        groups=base_groups,
        n_jobs=-1,
    )
    experiment_records.append({
        "feature_set": feature_set_name,
        "feature_count": X_experiment.shape[1],
        "cv_roc_auc_mean": scores.mean(),
        "cv_roc_auc_std": scores.std(),
    })
    fold_records.extend([
        {"feature_set": feature_set_name, "fold": index, "roc_auc": score}
        for index, score in enumerate(scores, start=1)
    ])

feature_engineering_results = pd.DataFrame(experiment_records)
base_score = feature_engineering_results.loc[
    feature_engineering_results["feature_set"].eq("Base"),
    "cv_roc_auc_mean",
].iloc[0]
feature_engineering_results["delta_vs_base"] = (
    feature_engineering_results["cv_roc_auc_mean"] - base_score
)
feature_engineering_results = feature_engineering_results.sort_values(
    "cv_roc_auc_mean", ascending=False
).reset_index(drop=True)
fold_results = pd.DataFrame(fold_records)

display(feature_engineering_results.round(6))
display(fold_results.pivot(index="fold", columns="feature_set", values="roc_auc").round(6))

,feature_set,feature_count,cv_roc_auc_mean,cv_roc_auc_std,delta_vs_base
0,Complete first batch,32,0.894308,0.003916,0.001819
1,Profile and history indicators,29,0.893998,0.003976,0.001508
2,Waiting-list interaction,27,0.892887,0.003575,0.000397
3,Base,25,0.892490,0.003574,0.000000
4,Stay composition,27,0.892486,0.003575,-0.000004


feature_set,Base,Complete first batch,Profile and history indicators,Stay composition,Waiting-list interaction
fold,,,,,
1,0.891752,0.893163,0.893016,0.891753,0.891916
2,0.889617,0.891913,0.891751,0.889611,0.890184
3,0.898706,0.901272,0.901022,0.898693,0.899246
4,0.893740,0.895359,0.894975,0.893754,0.893952
5,0.888633,0.889835,0.889227,0.888617,0.889136


## Criterio para interpretar los resultados

Una diferencia pequeña en la media no se considerará suficiente por sí sola. Se revisará si la mejora aparece de forma consistente en los folds, si aumenta la variabilidad y si la feature aporta una hipótesis interpretable. El resultado corresponde a Logistic Regression: una variable descartada para el modelo lineal todavía puede resultar útil en árboles.

Después de analizar esta tabla se decidirá qué features pasan al segundo lote. Sólo al cerrar la selección mediante CV se entrenará el conjunto elegido sobre todo 2015–2016 y se consultará el holdout 2017.